In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp, lit, input_file_name
)
from datetime import datetime

# ── Configuration ──────────────────────────────────────────
LAKEHOUSE_NAME  = "Interac_Bronze"
SHORTCUT_ROOT   = "Files/interac-raw/batch"
BRONZE_SCHEMA   = "bronze"

# Source file paths via shortcut
PATHS = {
    "transactions"     : f"{SHORTCUT_ROOT}/transactions/transactions.csv",
    "merchants"        : f"{SHORTCUT_ROOT}/merchants/merchants.csv",
    "cardholders"      : f"{SHORTCUT_ROOT}/cardholders/cardholders.csv",
    "interchange_fees" : f"{SHORTCUT_ROOT}/interchange_fees/interchange_fees.csv",
    "settlements"      : f"{SHORTCUT_ROOT}/settlements/settlements.csv",
    "disputes"         : f"{SHORTCUT_ROOT}/disputes/disputes.csv",
    "fraud_labels"     : f"{SHORTCUT_ROOT}/fraud_labels/fraud_labels.csv",
}

print("Configuration loaded successfully")
print(f"Ingestion started at: {datetime.now()}")

StatementMeta(, d378dd0d-f886-4fd4-891d-ca988eeb0905, 3, Finished, Available, Finished, False)

Configuration loaded successfully
Ingestion started at: 2026-05-05 22:20:27.201679


In [2]:
def ingest_csv_to_bronze(table_name, file_path):
    """
    Reads a CSV file from the OneLake shortcut and writes it
    as a Delta table in the Bronze lakehouse with audit columns.
    """
    print(f"\n{'='*60}")
    print(f"Ingesting: {table_name}")
    print(f"Source   : {file_path}")

    # Read CSV with header and schema inference
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("multiLine", "true")
          .option("escape", '"')
          .csv(file_path))

    # Add 5 audit metadata columns
    df_with_audit = (df
        .withColumn("_ingested_at",    current_timestamp())
        .withColumn("_source_file",    input_file_name())
        .withColumn("_pipeline_name",  lit("NB_01_Bronze_Ingestion"))
        .withColumn("_lakehouse",      lit(LAKEHOUSE_NAME))
        .withColumn("_batch_date",     lit(datetime.now().strftime("%Y-%m-%d")))
    )

    row_count = df_with_audit.count()

    # Write to Bronze Delta table
    (df_with_audit.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{table_name}"))

    print(f"Rows loaded : {row_count:,}")
    print(f"Delta table : {table_name}")
    print(f"Status      : SUCCESS")
    return row_count

StatementMeta(, d378dd0d-f886-4fd4-891d-ca988eeb0905, 4, Finished, Available, Finished, False)

In [3]:
# ── Run ingestion for all 7 tables ─────────────────────────
results = {}

for table_name, file_path in PATHS.items():
    try:
        count = ingest_csv_to_bronze(table_name, file_path)
        results[table_name] = {"status": "SUCCESS", "rows": count}
    except Exception as e:
        print(f"FAILED: {table_name} — {str(e)}")
        results[table_name] = {"status": "FAILED", "rows": 0}

StatementMeta(, d378dd0d-f886-4fd4-891d-ca988eeb0905, 5, Finished, Available, Finished, False)


Ingesting: transactions
Source   : Files/interac-raw/batch/transactions/transactions.csv
Rows loaded : 145,946
Delta table : transactions
Status      : SUCCESS

Ingesting: merchants
Source   : Files/interac-raw/batch/merchants/merchants.csv
Rows loaded : 2,000
Delta table : merchants
Status      : SUCCESS

Ingesting: cardholders
Source   : Files/interac-raw/batch/cardholders/cardholders.csv
Rows loaded : 5,000
Delta table : cardholders
Status      : SUCCESS

Ingesting: interchange_fees
Source   : Files/interac-raw/batch/interchange_fees/interchange_fees.csv
Rows loaded : 168
Delta table : interchange_fees
Status      : SUCCESS

Ingesting: settlements
Source   : Files/interac-raw/batch/settlements/settlements.csv
Rows loaded : 34,561
Delta table : settlements
Status      : SUCCESS

Ingesting: disputes
Source   : Files/interac-raw/batch/disputes/disputes.csv
Rows loaded : 3,500
Delta table : disputes
Status      : SUCCESS

Ingesting: fraud_labels
Source   : Files/interac-raw/batch/fraud

In [4]:
# ── Ingestion Summary ───────────────────────────────────────
print("\n" + "="*60)
print("BRONZE INGESTION SUMMARY")
print("="*60)

total_rows = 0
for table, result in results.items():
    status = result["status"]
    rows   = result["rows"]
    total_rows += rows
    print(f"{table:<25} {status:<10} {rows:>10,} rows")

print("="*60)
print(f"{'TOTAL':<25} {'':10} {total_rows:>10,} rows")
print(f"Completed at: {datetime.now()}")

StatementMeta(, d378dd0d-f886-4fd4-891d-ca988eeb0905, 6, Finished, Available, Finished, False)


BRONZE INGESTION SUMMARY
transactions              SUCCESS       145,946 rows
merchants                 SUCCESS         2,000 rows
cardholders               SUCCESS         5,000 rows
interchange_fees          SUCCESS           168 rows
settlements               SUCCESS        34,561 rows
disputes                  SUCCESS         3,500 rows
fraud_labels              SUCCESS         2,800 rows
TOTAL                                   193,975 rows
Completed at: 2026-05-05 22:21:35.781796
